In [ ]:
import os
import cv2
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

In [ ]:
def listar_arquivos(pasta: str):
    if not os.path.exists(pasta):
        print(f"A pasta '{pasta}' não existe.")
        return

    arquivos = [f for f in os.listdir(pasta) if os.path.isfile(os.path.join(pasta, f))]
    lista_arquivos = []
    for arquivo in arquivos:
        lista_arquivos.append(arquivo)
    if lista_arquivos:
        return lista_arquivos
    else:
        print(f"Nenhum arquivo encontrado na pasta '{pasta}'.")

In [ ]:
@dataclass
class AugmentConfig:
    # Flip horizontal
    p_flip: float = 0.5

    # Brightness jitter (ganho multiplicativo e offset aditivo)
    # Ex.: ganho em [1-brightness_gain, 1+brightness_gain]
    brightness_gain: float = 0.15
    # offset em [-brightness_offset, +brightness_offset] (em 0..255)
    brightness_offset: float = 12.0
    p_brightness: float = 0.9

    # Time-warp (suave, monotônico)
    # warp_strength controla o quanto “entorta” o tempo (0.0 = sem warp)
    warp_strength: float = 0.20
    p_time_warp: float = 0.7

    # Saída
    output_width: Optional[int] = None  # se quiser padronizar largura (ex. 640); mantém aspect ratio
    keep_fps: bool = True               # se False, você pode setar fps_out manualmente
    fps_out: Optional[float] = None

    # Se True, mantém exatamente o mesmo número de frames da entrada
    keep_num_frames: bool = True


class VideoAugmenter:
    """
    Aumentos de dados em vídeo:
      - flip horizontal
      - jitter de brilho
      - time-warp (distorção temporal suave e monotônica)

    Uso típico (Jupyter):
      aug = VideoAugmenter(seed=42)
      cfg = AugmentConfig(p_flip=1.0, p_time_warp=1.0, warp_strength=0.25, output_width=640)
      meta = aug.augment_video("in.mp4", "out.mp4", cfg)
      meta
    """

    def __init__(self, seed: Optional[int] = None):
        self.rng = np.random.default_rng(seed)

    # -----------------------------
    # IO
    # -----------------------------
    def read_video(self, path: str) -> Tuple[List[np.ndarray], Dict]:
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise FileNotFoundError(f"Não consegui abrir o vídeo: {path}")

        fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        frames = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(frame)

        cap.release()

        meta = {"fps": fps, "width": w, "height": h, "n_frames_declared": n, "n_frames_read": len(frames)}
        if len(frames) == 0:
            raise ValueError(f"Vídeo sem frames (ou codec inválido): {path}")
        return frames, meta

    def write_video(self, path: str, frames: List[np.ndarray], fps: float):
        h, w = frames[0].shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # boa compat. geral; troque se quiser (ex.: "avc1")
        out = cv2.VideoWriter(path, fourcc, float(fps), (w, h))
        if not out.isOpened():
            raise RuntimeError(f"Não consegui criar VideoWriter em: {path}")

        for f in frames:
            if f.shape[:2] != (h, w):
                raise ValueError("Todos os frames devem ter o mesmo tamanho.")
            out.write(f)
        out.release()

    # -----------------------------
    # Augmentations
    # -----------------------------
    def _maybe_resize(self, frames: List[np.ndarray], output_width: Optional[int]) -> List[np.ndarray]:
        if output_width is None:
            return frames
        h, w = frames[0].shape[:2]
        if w == output_width:
            return frames
        scale = output_width / float(w)
        new_h = int(round(h * scale))
        resized = [cv2.resize(f, (output_width, new_h), interpolation=cv2.INTER_AREA) for f in frames]
        return resized

    def _flip_horizontal(self, frames: List[np.ndarray]) -> List[np.ndarray]:
        return [cv2.flip(f, 1) for f in frames]

    def _brightness_jitter(self, frames: List[np.ndarray], gain: float, offset: float) -> List[np.ndarray]:
        # ganho multiplicativo e offset aditivo, ambos constantes no vídeo inteiro (coerência temporal)
        g = float(self.rng.uniform(1.0 - gain, 1.0 + gain))
        b = float(self.rng.uniform(-offset, +offset))

        out = []
        for f in frames:
            x = f.astype(np.float32) * g + b
            x = np.clip(x, 0, 255).astype(np.uint8)
            out.append(x)
        return out

    def _time_warp(self, frames: List[np.ndarray], strength: float, keep_len: bool = True) -> List[np.ndarray]:
        """
        Distorção temporal suave e monotônica.
        Ideia: criar uma função s(t) (0..1)->(0..1) que "entorta" o tempo,
        depois reamostrar frames usando interpolação linear no índice.
        """
        n = len(frames)
        if n < 4 or strength <= 1e-6:
            return frames

        # grade normalizada do tempo de saída
        t = np.linspace(0.0, 1.0, n, dtype=np.float32)

        # gera offsets suaves via soma de senóides (bem leve e monotônico após ajuste)
        # quanto maior strength, maior a curvatura
        k = int(self.rng.integers(1, 4))  # número de componentes
        phase = self.rng.uniform(0, 2*np.pi, size=k)
        freq = self.rng.uniform(0.5, 2.5, size=k)
        amp = self.rng.uniform(0.2, 1.0, size=k)

        smooth = np.zeros_like(t)
        for i in range(k):
            smooth += amp[i] * np.sin(2*np.pi*freq[i]*t + phase[i])

        # normaliza e aplica força
        smooth = smooth / (np.max(np.abs(smooth)) + 1e-6)
        s = t + (strength * 0.25) * smooth  # 0.25 pra não ficar agressivo demais

        # garante monotonicidade (cummax) e normaliza para [0,1]
        s = np.maximum.accumulate(s)
        s = (s - s[0]) / (s[-1] - s[0] + 1e-6)
        s = np.clip(s, 0.0, 1.0)

        # mapa para índices da entrada
        idx = s * (n - 1)

        # reamostra com interpolação linear em tempo (entre frames adjacentes)
        warped = []
        for x in idx:
            i0 = int(np.floor(x))
            i1 = min(i0 + 1, n - 1)
            a = float(x - i0)
            f0 = frames[i0].astype(np.float32)
            f1 = frames[i1].astype(np.float32)
            fw = (1.0 - a) * f0 + a * f1
            warped.append(np.clip(fw, 0, 255).astype(np.uint8))

        if keep_len:
            return warped
        return warped  # aqui manteria igual de qualquer forma (por construção)

    # -----------------------------
    # Pipeline
    # -----------------------------
    def augment_frames(self, frames: List[np.ndarray], cfg: AugmentConfig) -> Tuple[List[np.ndarray], Dict]:
        applied = {"flip": False, "brightness": False, "time_warp": False}

        # resize (se quiser padronizar antes)
        frames = self._maybe_resize(frames, cfg.output_width)

        # flip
        if self.rng.random() < cfg.p_flip:
            frames = self._flip_horizontal(frames)
            applied["flip"] = True

        # brightness jitter
        if self.rng.random() < cfg.p_brightness:
            frames = self._brightness_jitter(frames, cfg.brightness_gain, cfg.brightness_offset)
            applied["brightness"] = True

        # time-warp
        if self.rng.random() < cfg.p_time_warp:
            frames = self._time_warp(frames, cfg.warp_strength, keep_len=cfg.keep_num_frames)
            applied["time_warp"] = True

        return frames, applied

    def augment_video(self, input_path: str, output_path: str, cfg: AugmentConfig) -> Dict:
        frames, meta_in = self.read_video(input_path)

        frames_aug, applied = self.augment_frames(frames, cfg)

        # fps de saída
        fps = meta_in["fps"]
        if not cfg.keep_fps:
            if cfg.fps_out is None:
                raise ValueError("keep_fps=False, mas fps_out não foi definido.")
            fps = float(cfg.fps_out)

        self.write_video(output_path, frames_aug, fps=fps)

        meta_out = {
            "input": meta_in,
            "output": {
                "fps": fps,
                "width": frames_aug[0].shape[1],
                "height": frames_aug[0].shape[0],
                "n_frames": len(frames_aug),
                "applied": applied,
                "output_path": output_path,
            },
        }
        return meta_out


# -----------------------------
# EXEMPLO JUPYTER (descomente e rode)
# -----------------------------
# aug = VideoAugmenter(seed=123)
# cfg = AugmentConfig(
#     p_flip=0.5,
#     p_brightness=0.9,
#     brightness_gain=0.15,
#     brightness_offset=12,
#     p_time_warp=0.7,
#     warp_strength=0.2,
#     output_width=640
# )
# meta = aug.augment_video("entrada.mp4", "saida_aug.mp4", cfg)
# meta


In [ ]:
lista_arquivos = listar_arquivos("./entrada")

configs = {}
configs['flip'] = AugmentConfig(p_flip=1.0, p_brightness=0.0, p_time_warp=0.0)
configs['time_warp'] = AugmentConfig(p_flip=0.0, p_brightness=0.0, p_time_warp=1.0, warp_strength=0.25)
configs['brightness'] = AugmentConfig(p_flip=0.0, p_brightness=1.0, brightness_gain=0.15, brightness_offset=12, p_time_warp=0.0)
configs['combined'] = AugmentConfig(p_flip=0.5, p_brightness=0.9, brightness_gain=0.15, brightness_offset=12, p_time_warp=0.7, warp_strength=0.2, output_width=640)


for arquivo in lista_arquivos:
    for title, cfg in configs.items():
        aug = VideoAugmenter(seed=42)
        meta = aug.augment_video(os.path.join("entrada", arquivo), os.path.join("entrada/aug", f"aug_{title}_{arquivo}"), cfg)
        print(f"Processado {arquivo}: {meta['output']['applied']}")
